# Complete the four-model demo: original dots.mocr BF16
Run only the 46 missing IRENA/IMF pages. The existing NASA and handwriting outputs are reused.

Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. When prompted, select `four_model_demo_matched_inputs_46_pages.zip` from your Downloads folder.

The pinned original weights run entirely on GPU with BF16 storage and no quantization. Runtime, dtype, input hashes, failures and memory measurements are retained. BF16 arithmetic support is not inferred from storage dtype. Inputs contain no reference answers or API keys.

The result ZIP downloads automatically. Keep it in Downloads so it can be imported and checked locally. Do not publish the private matched input ZIP.

In [ ]:
# Only the 46 missing IRENA/IMF pages. NASA and handwriting are reused locally.
import hashlib
import pathlib
import shutil
import subprocess
import sys
import zipfile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime first"
assert torch.cuda.get_device_properties(0).total_memory >= 12 * 1024**3, "Use a >=12 GiB GPU"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.57.6",
        "accelerate>=1.10,<2",
        "pillow>=11,<13",
    ]
)
uploaded = files.upload()
assert len(uploaded) == 1, "Select exactly the prepared 46-page input ZIP"
name, payload = next(iter(uploaded.items()))
assert (
    hashlib.sha256(payload).hexdigest()
    == "497e2f1286a15fe6fcf09c971b5290f9e12e477cb983cf87a9ff78ca3580a2b9"
), "Input ZIP differs from the matched selection"
folder = pathlib.Path("/content/dots_mocr_demo_v5_inputs")
output = pathlib.Path("/content/dots_mocr_demo_v5_bf16_output")
assert not folder.exists() and not output.exists(), "Retain previous results before another run"
with zipfile.ZipFile(name) as archive:
    for member in archive.infolist():
        path = pathlib.PurePosixPath(member.filename)
        assert not path.is_absolute() and ".." not in path.parts
    archive.extractall(folder)
del uploaded, payload
subprocess.check_call(
    [
        sys.executable,
        "-u",
        str(folder / "run_gpu_test.py"),
        "--inputs",
        str(folder),
        "--output",
        str(output),
        "--weight-dtype",
        "bfloat16",
    ]
)
result = shutil.make_archive("/content/dots_mocr_unquantized_demo_46_pages_results", "zip", output)
print("All attempts saved; download the results ZIP:", result, flush=True)
files.download(result)